# COF-Landscaper local workflow

In [ ]:
import coflandscaper as cl

### Setup

`MODE` accepts `"serr"`, `"incl"`, or `"both"`. Use a distinct `COF_NAME` for each calculation.

Provide topology-appropriate `.xyz` building blocks in `0_node/` and `0_linker/`. Each intended connection site must be marked with `He`.

In [ ]:
TOPOLOGY = "hcb"
COF_NAME = "COF-LZU-1"
MODE = "both"

### Single-Layer COF Construction & Pre-Optimization

This step generates the single-layer COF structure and performs MACE pre-optimization.

> **Practical note:** If the input building-block fragments are strongly non-planar, the generated stacking matrix can contain strained or sterically congested geometries. Starting from relatively planar building units is therefore recommended, followed by relaxation of the assembled framework. Because the isolated single-layer model has more conformational freedom than the final stacked COF, the preoptimized layer may twist more strongly than the corresponding bulk structure.

In [ ]:
builder = cl.BuildCOF2D()
builder.build(topo=TOPOLOGY, cof_name=COF_NAME)

In [ ]:
preopt = cl.MaceOpt()
preopt.run_preopt(cof_name=COF_NAME)

### ILD × ILS Structure Matrix Generation

This step generates stacked COF structures by systematically varying:

- **Interlayer Distance (ILD)** (default range: 3.0-4.0 A in steps of 0.1 A)  
- **Interlayer Slipping (ILS)** (default range: 0.0 A to the AB limit in steps of 1.0 A)  
  Default ILS shift length and angle are auto-computed from the topology; use `print_shift=True` to display them.

The input structure is the pre-optimized single-layer (AA stacking with large interlayer distance).  
This step converts it into physically meaningful bulk stacking configurations.

Generated stacked structures organized by mode:  
- `.../serr/` → serrated stacking configurations  
- `.../incl/` → inclined stacking configurations  

These structures are intended for subsequent single-point energy evaluations and energy landscape analysis.


In [ ]:
matrix = cl.CreateMatrix()
matrix.run(cof_name=COF_NAME, topo=TOPOLOGY, mode=MODE)

### MACE Single-Point Energy Evaluation

This step computes single-point energies for generated structures using `MaceSP` (no geometry optimization).

Single-point energies are written ro `{COF_NAME}_sp_energies_{serr|incl}.csv` in `COF_NAME/3_{COF_NAME}_landscape/`  

In [ ]:
sp = cl.MaceSP()
sp.run(cof_name=COF_NAME, mode=MODE)

### Potential Energy Landscape (PES)

This step plots an approximate stacking potential energy surface by mapping the relative energies of the generated ILD/ILS structure matrix as a function of interlayer distance (ILD) and interlayer slipping (ILS).

Because no full structural relaxation is performed at this stage, the PES is treated as a reduced-dimensional screening model rather than a full description of the underlying high-dimensional energy landscape. Within this approximation, it provides a qualitative to semi-quantitative representation of the relative stability of different stacking arrangements and serves to identify candidate minima for subsequent refinement by full geometry optimization.

---

- `show` *(bool, optional, default: `False`)*  
  If `True`, displays interactive plot windows.  

- PES plots `pes_{COF_NAME}_{serr|incl}.png` default written to:  
  `COF_NAME/3_{COF_NAME}_landscape/`

In [ ]:
landscape = cl.Landscape()
landscape.run(cof_name=COF_NAME, mode=MODE, show=True)

### Structure Selection for Optimization

This step selects candidate structures corresponding to automatically detected global or local minima and copies them into dedicated folders for subsequent geometry optimization.

By default, the global minimum of each stacking mode is selected automatically. Local minima can instead be selected by setting the corresponding minima option.

Additional sampling points identified from inspection of the simplified PES can also be included by providing `(ILD, ILS)` pairs separately for the serrated and inclined stacking modes, for example:

```python
EXTRA_SERR = [(3.4, 2.0), (3.5, 1.0)]
EXTRA_INCL = [(3.3, 2.0)]

In [ ]:
EXTRA_SERR = []
EXTRA_INCL = []

In [ ]:
selector = cl.SelectCofs()
selector.run(
    cof_name=COF_NAME,
    mode=MODE,
    selections_serr=EXTRA_SERR,
    selections_incl=EXTRA_INCL,
)

### MACE Geometry Optimization

This step performs geometry optimizations of selected stacking structures using `MaceOpt`.

`MaceOpt` uses ASE `FrechetCellFilter` + `LBFGS`, writes optimized CIF files, and can write a combined energy CSV (`{COF_NAME}_opt_energies_per_layer.csv`).

  Optimized structures (`.cif`) and optional energies CSV are written to:  
  `COF_NAME/4_{COF_NAME}_optimization/`

In [ ]:
opt = cl.MaceOpt()
opt.run(cof_name=COF_NAME, mode=MODE)

### Analysis & Visualization

This step analyzes optimized structures by computing interlayer distance (ILD) and interlayer slipping (ILS) from the optimized structures, and writing a summary CSV for comparison across stacking configurations.

In addition, structures can be visualized using an interactive viewer.

In [ ]:
# Defaults
analyzer = cl.AnalyzeStacking()
analyzer.analyze(cof_name=COF_NAME, mode=MODE)

In [ ]:
# Defaults
visualizer = cl.VisualizeCOF()
visualizer.visualize_cof(cof_name=COF_NAME, mode=MODE)

### PXRD simulation

Simulate PXRD patterns from the optimized structures. The simulated `.xy` patterns are written to the analysis-stage `pxrd_xy` folders.

In [ ]:
pxrd = cl.PXRD(wavelength="CuKa", two_theta_range=(1.5, 30.0))
pxrd.run(cof_name=COF_NAME, mode=MODE)

#### Simulated peak tables

Extract simulated reflections (100 by default) per structure . The resulting peak lists are saved as `<structure>_all.csv` in the mode-specific `pxrd_peaks` folders.

In [ ]:
pxrd.extract_peaks(cof_name=COF_NAME, mode=MODE)

### Experimental comparison

Place exactly one experimental `.xy` file in `experimental_pxrd/`, or pass its path explicitly. The comparison generates one PDF per simulated structure in `pxrd_plots`.

In [ ]:
pxrd.plot_sim_vs_exp(
    cof_name=COF_NAME,
    mode=MODE,
    xlim=(3, 30),
)

## PXRD-guided refinement

The simulated and experimental PXRD patterns should first be inspected qualitatively. The refinement described below is intended for structures that already reproduce the experimental pattern reasonably well but show a **systematic offset in peak positions**. It is not intended to correct a stacking model whose simulated PXRD pattern is fundamentally inconsistent with experiment.

A consistent displacement of corresponding simulated and experimental reflections can indicate that the predicted structure is qualitatively correct while its unit-cell dimensions differ slightly from those of the experimental material. COF-Landscaper therefore provides a simple PXRD-guided refinement in which experimental diffraction peak positions are used to adjust the in-plane dimensions of the calculated structure. This follows the general principle of refining lattice dimensions against diffraction data.

If both serrated and inclined structures were evaluated above, first select the stacking mode(s) that provides the physically meaningful agreement with experiment. The comparison is performed within **user-defined 2θ regions**. Each region should contain an experimental feature and the corresponding simulated reflection that the user intends to compare.

A region is specified by its lower and upper 2θ limits:
Example:
```python

peak_regions = [
    (4.0, 5.5),
    (7.5, 9.0),
    (9.0, 10.5),
]

In [ ]:
MODE = "serr"

In [ ]:
peak_regions = [
    (4.0, 5.5),
    (7.5, 9),
    (9, 10.5),
    (12, 14),
    (15.5, 17.5),
    (24, 27.5),
]

pxrd.extract_peak_regions(
    cof_name=COF_NAME,
    mode=MODE,
    peak_regions=peak_regions,
)

### In-plane lattice scaling

The extracted experimental and simulated peak positions are used to estimate the in-plane lattice correction required to improve their agreement.

By default, all extracted peak regions are included (`scale_regions=None`). If some experimental features are broad, overlapping, or otherwise unsuitable for a reliable comparison, `scale_regions` can be used to restrict the scaling to selected regions.

For example, to use only regions 1, 2, and 4:

```python
scale_regions = [1, 2, 4]

In [ ]:
scaled_cifs = pxrd.generate_scaled_cif(
    cof_name=COF_NAME,
    mode=MODE,
    scale_regions=[1,2],
)

### Post-optimization

The PXRD-guided scaling changes the unit-cell dimensions geometrically. The scaled structure is therefore subsequently re-optimized with MACE while keeping the refined unit cell fixed, allowing the atomic coordinates to relax within the experimentally informed cell.

In [ ]:
postopt = cl.MaceOpt()
postopt.run_fixed(cof_name=COF_NAME, mode=MODE)

## Postopt analysis

Repeat structural and PXRD analysis against the post-optimized structures using `source="postopt"`.

In [ ]:
analyzer = cl.AnalyzeStacking()
analyzer.analyze(cof_name=COF_NAME, mode=MODE, source="postopt")

In [ ]:
pxrd.run(cof_name=COF_NAME, mode=MODE, source="postopt")
pxrd.extract_peaks(cof_name=COF_NAME, mode=MODE, source="postopt")
pxrd.extract_peak_regions(
    cof_name=COF_NAME,
    mode=MODE,
    peak_regions=peak_regions,
    source="postopt",
)

In [ ]:
pxrd.plot_sim_vs_exp(
    cof_name=COF_NAME,
    mode=MODE,
    source="postopt",
    xlim=(3.5, 30.0),
)

In [ ]:
visualizer = cl.VisualizeCOF()
visualizer.visualize_cof(
    cof_name=COF_NAME,
    mode=MODE,
    source="postopt",
)